# Build Vector Store

Use LlamaIndex to build a data stores for the preprocessed HTML payloads.

A few options have been tested as follows:
1. normal vector store
2. FAISS vector store

In [3]:
import os
import numpy as np
import pandas as pd
from llama_index.core import Document, VectorStoreIndex, Settings, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


embedding_model = "all-MiniLM-L6-v2"  # Pre-trained embedding model
preprocessed_corpus_path = "./sample_data/preprocessed_corpus.parquet"

# ---------------------------
# 1. Load Model
# ---------------------------
Settings.llm = None
Settings.embed_model = HuggingFaceEmbedding(model_name=embedding_model)

# ----------------------------
# 2. Create Documents based on the loaded data
# ----------------------------
documents = [
    Document(
        text=row.text,
        metadata={
            "snapshot_id": str(row.snapshot_id),
            "page_idx": int(row.page_idx),
            "url": row.url,
        },
        excluded_llm_metadata_keys=["url"],
    )
    for row in pd.read_parquet(
        preprocessed_corpus_path,
        columns=["text", "snapshot_id", "page_idx", "url"],
    ).itertuples(index=False)
]

LLM is explicitly disabled. Using MockLLM.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Default vector store

Default in-memory vector store built into LlamaIndex, including

- Default Document Store (docstore)
- Default Vector Store (default__vector_store)
- Default Index (index_store)

It is essentially a Python-based vector database that:

- stores embeddings in lists / dicts
- performs similarity search using NumPy
- keeps everything in memory

In [5]:
# ----------------------------
# Construct Index
# ----------------------------
index = VectorStoreIndex.from_documents(documents)
print("Indexed documents:", len(documents))

# ----------------------------
# Save index to disk
# ----------------------------
persist_dir = "./sample_data/vector_store/default"
os.makedirs(persist_dir, exist_ok=True)
index.storage_context.persist(persist_dir=persist_dir)

print(f"Vector store saved to {persist_dir}")

Indexed documents: 2204
Vector store saved to ./sample_data/vector_store/default


## FAISS vector store

A FAISS store in LlamaIndex is a wrapper around the FAISS (Facebook AI Similarity Search) library that acts as a vector database for storing and retrieving embeddings.

It provides:

- Efficient similarity search using FAISS algorithms (e.g., L2, cosine, IVF, HNSW)
- Fast nearest-neighbor lookup even for millions of vectors
- A connection layer between LlamaIndex’s documents/nodes and FAISS’s raw vector index

What it stores:

- The embedding vectors
- IDs that link vectors to LlamaIndex nodes/documents

What it does not store:

- Raw text
- Metadata
- Chunked nodes

Those are handled by LlamaIndex’s docstore, not FAISS.

In [6]:
# Add additional import
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore


# Get embedding dimension from the model
embed_model = Settings.embed_model
# Create a sample embedding to get the dimension
sample_embedding = embed_model.get_text_embedding("sample")
embedding_dim = len(sample_embedding)

In [7]:
# ----------------------------
# Create FAISS Index
# ----------------------------
# Create a FAISS index using L2 (Euclidean) distance
faiss_index = faiss.IndexFlatL2(embedding_dim)

# Wrap FAISS index with LlamaIndex's FaissVectorStore
vector_store = FaissVectorStore(faiss_index=faiss_index)

# Create storage context with FAISS vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# ----------------------------
# Construct Index with FAISS
# ----------------------------
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print("Indexed documents:", len(documents))

# ----------------------------
# Save index to disk
# ----------------------------
persist_dir = "./sample_data/vector_store/faiss"
os.makedirs(persist_dir, exist_ok=True)
index.storage_context.persist(persist_dir=persist_dir)

print(f"FAISS vector store saved to {persist_dir}")

Indexed documents: 2204
FAISS vector store saved to ./sample_data/vector_store/faiss


# Semantic Search Pipeline

The following cells demonstrate how to perform semantic search using the prepared vector stores.

**Features**
- Load persisted vector stores (default and FAISS)
- Configure query engines with different parameters
- Run semantic search queries
- Compare results and performance

## 1. Setup and Imports

In [ ]:
import os
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from IPython.display import Markdown, display
import time

## 2. Configure Embedding Model

In [ ]:
# Disable default LLM (we only need embeddings for semantic search)
Settings.llm = None

# Load the same embedding model used during indexing
embedding_model = "all-MiniLM-L6-v2"
Settings.embed_model = HuggingFaceEmbedding(model_name=embedding_model)

print(f"Embedding model loaded: {embedding_model}")

LLM is explicitly disabled. Using MockLLM.


/home/yzhan/miniconda3/envs/env_nlnz/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded: all-MiniLM-L6-v2


## 3. Load Default Vector Store

In [ ]:
persist_dir_default = "./sample_data/vector_store"

if os.path.exists(persist_dir_default):
    storage_context_default = StorageContext.from_defaults(
        persist_dir=persist_dir_default
    )
    index_default = load_index_from_storage(storage_context_default)
    print(f"✓ Default vector store loaded from {persist_dir_default}")
else:
    print(f"✗ Default vector store not found at {persist_dir_default}")
    index_default = None

✓ Default vector store loaded from ./sample_data/vector_store


## 4. Load FAISS Vector Store

In [ ]:
persist_dir_faiss = "./sample_data/vector_store_faiss"

if os.path.exists(persist_dir_faiss):
    try:
        from llama_index.vector_stores.faiss import FaissVectorStore
        import faiss

        # FAISS stores the index as binary data in default__vector_store.json
        # We need to load it using faiss.read_index and then reconstruct the vector store
        faiss_index_path = os.path.join(persist_dir_faiss, "default__vector_store.json")

        if os.path.exists(faiss_index_path):
            print(f"Loading FAISS index from: {faiss_index_path}")

            # Load the FAISS index from the binary file
            faiss_index = faiss.read_index(faiss_index_path)
            print(f"✓ FAISS index loaded: {faiss_index.ntotal} vectors")

            # Reconstruct the FaissVectorStore
            vector_store_faiss = FaissVectorStore(faiss_index=faiss_index)

            # Load the storage context with the reconstructed vector store
            storage_context_faiss = StorageContext.from_defaults(
                vector_store=vector_store_faiss, persist_dir=persist_dir_faiss
            )

            # Load the index
            index_faiss = load_index_from_storage(storage_context_faiss)
            print(f"✓ FAISS vector store loaded successfully from {persist_dir_faiss}")
    except ImportError as e:
        print(f"✗ FAISS dependencies not installed: {e}")
        index_faiss = None
    except Exception as e:
        print(f"✗ Error loading FAISS vector store: {e}")
        print(f"  Error type: {type(e).__name__}")
        import traceback

        traceback.print_exc()
        index_faiss = None
else:
    print(f"✗ FAISS vector store not found at {persist_dir_faiss}")
    index_faiss = None

Loading FAISS index from: ./sample_data/vector_store_faiss/default__vector_store.json
✓ FAISS index loaded: 3345 vectors
✓ FAISS vector store loaded successfully from ./sample_data/vector_store_faiss


## 5. Create Query Engines

Query engines handle the semantic search process. Key parameters:
- `similarity_top_k`: Number of most similar documents to retrieve
- `response_mode`: How to synthesize the response (we'll use 'no_text' to just get documents)

In [ ]:
# Create query engines with different configurations
query_engines = {}

if index_default:
    query_engines["default"] = index_default.as_query_engine(
        similarity_top_k=5,
        response_mode="no_text",  # Only retrieve documents, no LLM synthesis
    )
    print("✓ Default query engine created (top_k=5)")

if index_faiss:
    query_engines["faiss"] = index_faiss.as_query_engine(
        similarity_top_k=5, response_mode="no_text"
    )
    print("✓ FAISS query engine created (top_k=5)")

print(f"\nAvailable query engines: {list(query_engines.keys())}")

✓ Default query engine created (top_k=5)
✓ FAISS query engine created (top_k=5)

Available query engines: ['default', 'faiss']


## 6. Helper Functions for Result Display

In [ ]:
def display_search_results(query, response, engine_name=""):
    """
    Display semantic search results in a formatted way.
    """
    title = (
        f"### Search Results: {engine_name}" if engine_name else "### Search Results"
    )
    display(Markdown(title))
    display(Markdown(f"**Query:** {query}"))
    display(Markdown(f"**Results Found:** {len(response.source_nodes)}"))
    display(Markdown("---"))

    for i, node in enumerate(response.source_nodes, start=1):
        # Extract metadata
        metadata = node.node.metadata
        score = node.score if hasattr(node, "score") else "N/A"

        # Display result
        display(
            Markdown(
                f"#### Result #{i} (Score: {score:.4f})"
                if isinstance(score, float)
                else f"#### Result #{i}"
            )
        )
        display(Markdown(f"**URL:** {metadata.get('url', 'N/A')}"))
        display(
            Markdown(
                f"**Snapshot ID:** {metadata.get('snapshot_id', 'N/A')}, **Page Index:** {metadata.get('page_idx', 'N/A')}"
            )
        )

        # Show text preview (first 300 characters)
        text_preview = (
            node.node.text[:300] + "..."
            if len(node.node.text) > 300
            else node.node.text
        )
        display(Markdown(f"**Text Preview:**\n> {text_preview}"))
        display(Markdown("---"))


def compare_search_results(query, engines_dict):
    """
    Run the same query on multiple engines and compare results.
    """
    display(Markdown(f"## Comparing Query: '{query}'"))
    display(Markdown("---"))

    results = {}
    timings = {}

    for engine_name, engine in engines_dict.items():
        start_time = time.time()
        response = engine.query(query)
        elapsed_time = time.time() - start_time

        results[engine_name] = response
        timings[engine_name] = elapsed_time

        display_search_results(query, response, engine_name.upper())

    # Display timing comparison
    display(Markdown("### Performance Comparison"))
    for engine_name, elapsed in timings.items():
        display(Markdown(f"- **{engine_name.upper()}**: {elapsed:.4f} seconds"))

    return results, timings


print("Helper functions loaded successfully!")

Helper functions loaded successfully!


## 7. Example Queries

Let's run some semantic search queries to test the pipeline.

### Query 1: Traffic Light System

In [ ]:
query1 = "What is the traffic light system?"

# Use default engine if available
if "default" in query_engines:
    response1 = query_engines["default"].query(query1)
    display_search_results(query1, response1, "Default Vector Store")
else:
    print("Default vector store not available")

### Search Results: Default Vector Store

**Query:** What is the traffic light system?

**Results Found:** 5

---

#### Result #1 (Score: 0.5028)

...

#### Result #5 (Score: 0.3484)

**URL:** https://ndhadeliver.natlib.govt.nz/webarchive/20211213040406/https://covid19.govt.nz/traffic-lights/life-at-red/education-at-red/school-settings-until-1-january-2022/

**Snapshot ID:** 19, **Page Index:** 343

**Text Preview:**
> Home Traffic lights Life at Red Education School settings until 1 January 2022 Schools will move to the COVID-19 Protection Framework (traffic lights) from 1 January 2022. --- Section Separator --- Schools in Auckland will continue to follow Alert Level 3 Step 2 settings until 1 January 2022 when yo...

---

### Query 2: Vaccination Requirements

In [ ]:
query2 = "vaccination requirements and mandates"

if "default" in query_engines:
    response2 = query_engines["default"].query(query2)
    display_search_results(query2, response2, "Default Vector Store")
else:
    print("Default vector store not available")

### Search Results: Default Vector Store

**Query:** vaccination requirements and mandates

**Results Found:** 5

---

#### Result #1 (Score: 0.6611)

...

#### Result #5 (Score: 0.6435)

**URL:** https://ndhadeliver.natlib.govt.nz/webarchive/20221017033101/https://covid19.govt.nz/covid-19-vaccines/vaccinations-and-work/

**Snapshot ID:** 29, **Page Index:** 42

**Text Preview:**
> Home COVID-19 vaccines Vaccinations and work --- Section Separator --- The final mandate for government workers ended at 11:59 pm on Monday 26 September 2022. This applied to some health and disability workers. Some employers may still require workers to be vaccinated due to health and safety. We ha...

---

### Query 3: Face Mask Rules

In [ ]:
query3 = "face mask requirements at different alert levels"

if "default" in query_engines:
    response3 = query_engines["default"].query(query3)
    display_search_results(query3, response3, "Default Vector Store")
else:
    print("Default vector store not available")

### Search Results: Default Vector Store

**Query:** face mask requirements at different alert levels

**Results Found:** 5

---

#### Result #1 (Score: 0.6839)

...

#### Result #5 (Score: 0.5823)

**URL:** https://ndhadeliver.natlib.govt.nz/webarchive/20221017032051/https://covid19.govt.nz/prepare-and-stay-safe/keep-up-healthy-habits/face-masks/wearing-a-face-mask/

**Snapshot ID:** 29, **Page Index:** 224

**Text Preview:**
> If you cannot wear a face mask, you can apply for a Mask Exemption Pass. It can help make it easier to explain that a face mask is unsuitable for you. You do not have to show a Mask Exemption Pass — but it may help you feel more comfortable. Apply for a Mask Exemption Pass (external link) --- Sectio...

---

## 8. Compare Vector Stores

If both vector stores are available, let's compare their performance and results.

In [ ]:
if len(query_engines) > 1:
    comparison_query = "COVID-19 protection framework and alert levels"
    results, timings = compare_search_results(comparison_query, query_engines)
else:
    print(
        f"Only {len(query_engines)} query engine(s) available. Need at least 2 for comparison."
    )
    print(f"Available: {list(query_engines.keys())}")

## Comparing Query: 'COVID-19 protection framework and alert levels'

---

### Search Results: DEFAULT

**Query:** COVID-19 protection framework and alert levels

**Results Found:** 5

...

**Text Preview:**
> As restrictions have been relaxed in other countries around the world, the virus has had the opportunity to bounce back, and in some places it has. Ultimately, we need to stay in control. So the key for us has always been to understand where we are at in any given time in our battle with COVID and t...

---

### Performance Comparison

- **DEFAULT**: 0.0453 seconds

- **FAISS**: 0.0124 seconds